This notebook uses the following inputs:

- Projected socioeconomic indicators for SSP1-5, saved as geopackage files (output from step 3: **socioecon_projection**)
- Global base year socioeconomic and electricity-related indicators, saved as an Excel workbook
- SSP-consistent electricity access projections, saved as an Excel workbook

in order to:
1. Estimate the residential electricity demand of each populated cell via a sigmoid correlation function
2. Update the electrification status of each populated cell based on electricity access projections and prioritising currently unelectrified cells with higher income (i.e. higher ability to pay)

The resulting datasets are saved as shapefiles that can be used for analysis and result visualisation, as well as inputs to energy modelling or electrification planning tools (e.g. OnSSET).

## 1. Importing required packages

The following cell needs to be run first whenever the kernel is restarted.

In [ ]:
from geocube.vector import vectorize

import geopandas as gpd

import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.cm as cm

import numpy as np

import pandas as pd

import rasterio
from rasterio.enums import Resampling
from rasterio.warp import calculate_default_transform, reproject, Resampling
from rasterio.plot import show
from rasterio.features import rasterize
from rasterio.transform import from_origin

import rioxarray

from rtree import index

from scipy.optimize import curve_fit

from shapely.ops import nearest_points
from shapely.geometry import box

from sklearn.metrics import r2_score

import mapclassify

## 2. Country list and input files

In [ ]:
# User-defined country code and country name dictionary
ccode_dict = {'NAM': 'Namibia'}

# SSP scenarios and transition period
n_scenario = 5 # number of SSPs
start_year = 2025
end_year = 2050
year_int = 5 # length of each time step between the start and end years

In [ ]:
# Import required input files
# National socioeconomic and electricity-related indicators in 2015, from the SSP, World Bank and IEA
# incl. pop, gdp, inc, elec access, residential elec demand
socioecon_base = pd.read_excel('input/socioeconomic_data_ssp_iea_2015.xlsx', sheet_name='Data')

# SSP-consistent electricity access rate projections from Huisman et al. (Global Vulnerability Index Projections, GVIP)
elec_access_proj = pd.read_excel('input/ElecAccess_projection_GVIP.xlsx', sheet_name='GVIP_Database')  

In [ ]:
# Import high-resolution projected socioeconomic indicators (output of socioecon_projection)
for ccode in ccode_dict:
    for n in range(1, n_scenario + 1):
        socioecon_proj_name = ccode + '_socioecon_SSP' + str(n)
        file_path = '3_output/' + ccode +'_socioecon_SSP' + str(n) + '.gpkg'
        
        locals()[socioecon_proj_name] = gpd.read_file(file_path)

# display one geodataframe for reference
NAM_socioecon_SSP1

## 3. Residential electricity demand projection

### 3.1 Electricity demand estimation

The purpose of this step is to estimate the residential electricity demand of each populated cell, irrespective of its initial electrification status. The following steps apply:
1. A sigmoid function (S-Curve) is fitted to global income and residential electricity demand statistics in the base year.
2. The fitted correlation is used to estimate the average demand of each cell based on its income level.
3. Each cell is classified according to the Multi-Tier Framework based on its average demand.
4. The total demand is obtained by multiplying the average demand with the population count.

In [ ]:
#Correlation between electricity demand and income level based on global statistics using a sigmoid function
# defining a sigmoid function S(x) = min + (max-min) * {(1 /(1+exp(-k(x-x0)))^a}
min_e = np.log10(0.9) # min electricity demand = 0.9 kWh/capita/year, equivalent to MTF tier 1
max_e = np.log10(7500) # max electricity demand = 7,500 kWh/capita/year, equivalent to highest global value (Norway)

def sigmoid(x, k, a, x0):
    return 10 ** (min_e + (max_e - min_e) * (1/(1 + np.exp(-k * (np.log10(x)-np.log10(x0))))**a))

# providing an initial guess for the parameters (coefficients)
initial_guess = [1.5, 0.5, 26000]

# fitting the two-term exponential model
x_data = socioecon_base['INC_2017USD_calc']
y_data = socioecon_base['Res_Elec_kWhCapita']
params, _ = curve_fit(sigmoid, x_data, y_data, p0=initial_guess, maxfev=100000)

# displaying the sigmoid equation
k, a, x0 = params
print(f"x0 = {x0:,.0f}, k = {k:.1f}, a = {a:.1f}")

# calculating the predicted values
y_pred = sigmoid(x_data, *params)

# calculating R-squared value, as an indicator for model fitness
r_squared = r2_score(y_data, y_pred)
print(f"R-squared: {r_squared:.2f}")

# plotting the data and the two-term exponential fit on a logarithmic scale
fig, ax = plt.subplots()
plt.scatter(x_data, y_data, label='Data', s=100, c='yellow') # original data points

# adding labels to data points
for i, txt in enumerate(socioecon_base['Code']):
    ax.annotate(txt, (x_data.iat[i], y_data.iat[i]), fontsize=6, alpha=1, ha='center', va='center')

# s-curve trendline
x_fit = np.linspace(100, 150000, 10000)
y_fit = sigmoid(x_fit, *params)

# plotting trendline and legend
plt.plot(x_fit, y_fit, color='red',
         label=f"$x_{0}$ = {x0:,.0f}, k = {k:.1f}, a = {a:.1f}\n$R^{2}$ value = {r_squared:.2f}")
plt.xscale('log') # x-axis scale
plt.yscale('log') # y-axis scale
plt.xlabel('Income level, PPP [const. intl. 2017 USD/cap]') # x-axis label
plt.ylabel('Residential electricity demand [kWh/cap]') # y-axis label
plt.ylim(top=10000) # upper limit of y-axis range
plt.legend()
plt.grid(which='both', alpha=0.25) # show gridlines
plt.rcParams['font.family'] = 'sans-serif' # font type
plt.rcParams['font.size'] = 7 # font size
plt.savefig('4_output/INC_ResElec_corr_sigmoid.png', dpi=300) # saving the generated plot: file name and resolution
plt.show()

# displaying GDP and electricity consumption stats table for reference
socioecon_base.head()

In [ ]:
# Multi-Tier Framework classification
mtf_tiers_hh = [4.5, 73, 365, 1250, 3000] # in kWh/hh/year, annual household consumption, 5-person household (reference)
hh_size = 5 # reference household

# calculating equivalent per capita consumption based on household size
mtf_tiers_cap = [tier / hh_size for tier in mtf_tiers_hh] # in kWh/cap/year, annual per capita consumption

# printing resulting MTF classes for the study region, change labels as needed
print(f'Multi-Tier Framework classification based on reference household size:\nTier 1 >= {mtf_tiers_cap[0]:.1f} kWh/cap/year\nTier 2 >= {mtf_tiers_cap[1]:.0f} kWh/cap/year\nTier 3 >= {mtf_tiers_cap[2]:.0f} kWh/cap/year\nTier 4 >= {mtf_tiers_cap[3]:.0f} kWh/cap/year\nTier 5 >= {mtf_tiers_cap[4]:.0f} kWh/cap/year')

# Min. electricity demand to achieve universal electricity access per MTF definition
min_elec = mtf_tiers_cap[0]

# Max. electricity demand capped at base year highest observed consumption, i.e. that of Norway
max_elec = 7500

# creating bins and labels for each Tier for later electricity consumption classification
mtf_bins = mtf_tiers_cap.copy()
mtf_bins.append(max_elec)
labels = [1, 2, 3, 4, 5]

In [ ]:
# calculating per capita and total electricity consumption for each cell & MTF class
for ccode in ccode_dict:
    for n in range(1, n_scenario + 1):
        socioecon_proj_name = ccode + '_socioecon_SSP' + str(n)
        
        for x in range(start_year, end_year + year_int, year_int):
            elec_proj_name = ccode + '_elec_SSP' + str(n) + '_' + str(x)
            
            # retrieving relevant scocioeconomic indicators columns
            locals()[elec_proj_name] = locals()[socioecon_proj_name][['geometry', 'elec_stat', str(x) + '_POP', str(x) + '_INC']].copy()
            
            # estimating per capita electricity demand based on income level using sigmoid function
            locals()[elec_proj_name][str(x) + '_perCapitaElec'] = sigmoid(locals()[elec_proj_name][str(x) + '_INC'], *params)
            
            # calculating total demand per cell based on population count
            locals()[elec_proj_name][str(x) + '_TotElec'] = locals()[elec_proj_name][str(x) + '_perCapitaElec'] * locals()[elec_proj_name][str(x) + '_POP']
            
            # classifying each cell according to MTF based on per capita demand level
            locals()[elec_proj_name][str(x) + '_MTF'] = pd.cut(locals()[elec_proj_name][str(x) + '_perCapitaElec'], bins=mtf_bins, labels=labels, right=False)
    
            # renaming elec_stat column to specify year
            locals()[elec_proj_name].rename(columns={'elec_stat': '2015_elec'}, inplace=True)

# displaying one of the resulting GeoDataFrames
NAM_elec_SSP2_2030

### 3.2 Electricity access roll-out

The purpose of this step is to progressively update the initial electrification status of each populated cell. This is done based on the national-level electricity access projections from Huisman et al. (GVIP). Priority is given to currently unelectrified cells with higher income levels, which implies higher ability to pay. Other roll-out rules can be used as well.

N.B. The electricity access projections from Huisman et al. are based on crude assumptions, such that all countries are assumed to reach universal access by 2040 in SSP1 (optimistic, ~SSP5), 2050 in SSP2 (BAU), and 2060 in SSP3 (pessimistic, ~SSP4). Since not all countries have the same starting electricity access rate in the base year, this results in substantially different growth rates, some of which may not be realistic. It is recommended to align the growth rate in SSP2 with historical trends. Subsequently, SSPs 1&5 can be assumed to be ahead by one decade, and SSPs 3&4 to be behind by one decade.

In [ ]:
# updating the electricity access status of each cell based on electrification projections
# priority is given to cells with higher income; i.e. cells with higher ability to pay
for ccode in ccode_dict:
    for n in range(1, n_scenario + 1):
        for x in range(start_year, end_year + year_int, year_int):
            elec_proj_name = ccode + '_elec_SSP' + str(n) + '_' + str(x)
            prev_elec_name = ccode + '_elec_SSP' + str(n) + '_' + str(x-5)
    
            # retrieving projected electricity access rate for each scenario and year
            elec_access = elec_access_proj.loc[(elec_access_proj['iso_code'] == ccode) & (elec_access_proj['ssp'] == n) & (elec_access_proj['year'] == x), 'electr'].values[0] / 100
    
            # calculating total population for each scenario and year
            tot_pop = locals()[elec_proj_name][str(x)+'_POP'].sum()
    
            # arranging cells descendingly according to their income level
            locals()[elec_proj_name] = locals()[elec_proj_name].sort_values(by=[str(x) + '_INC'], ascending=False, ignore_index=True)
    
            # retrieving the electrification status of each cell from the previous modelled year
            if x == 2025:
                locals()[elec_proj_name]['prev_elec'] = locals()[elec_proj_name]['2015_elec']
            else:
                locals()[elec_proj_name]['prev_elec'] = locals()[prev_elec_name][str(x-5) + '_elec']
    
            locals()[elec_proj_name][str(x) + '_elec'] = locals()[elec_proj_name]['prev_elec']
            
            # classifying cells according to their previous electrification status:
            # class 0: previously un-electrified cells,
            # class 1: previously electrified cells, and
            # class 2: newly electrified cells
            locals()[elec_proj_name]['elec_class'] = np.where(locals()[elec_proj_name]['prev_elec'] == True, 1, 0)
            
            if elec_access == 1:
                # changing electrification status of all cells to true if projected access rate is 100%
                locals()[elec_proj_name][str(x) + '_elec'] = True
                locals()[elec_proj_name]['elec_class'] = np.where(locals()[elec_proj_name]['prev_elec'] == False, 2, 1)
                
                print('Status report for', ccode_dict[ccode], 'SSP'+str(n), 'in', str(x) + ':')
                print('All populated cells are classified as electrified since electricity access rate is equal to ' + f'{elec_access:.0%}')
            
            else:
                # incrementally change electrification status of cells to true until projected access rate is reached,
                # higher income cells get electrified first
                for r in range(0, len(locals()[elec_proj_name])):
                    locals()[elec_proj_name].at[r, str(x) + '_elec'] = True
                    locals()[elec_proj_name].at[r, 'elec_class'] = 2 if locals()[elec_proj_name].at[r, 'prev_elec'] == False else 1
                    
                    # re-calculating electricity access rate after each iteration
                    elec_access_calc = locals()[elec_proj_name].loc[locals()[elec_proj_name][str(x) + '_elec'] == True, str(x)+'_POP'].sum() / tot_pop
                    
                    if elec_access_calc >= elec_access:
                        # exiting the loop when projected access rate is reached,
                        # printing resulting access rate
                        print('Status report for', ccode_dict[ccode], 'SSP'+str(n), 'in', str(x) + ':')
                        print('Calculated electricity access rate: ', f'{(elec_access_calc):.3%}')
                        break

### 3.3 Saving electricity demand projections

The purpose of this step is to save the resulting electricity demand and electricity access roll-out projections. These are saved as shapefiles for each SSP and year, as well as raster files for each of the indicators: electricity demand per capita, total electricity demand, MTF classification, and electrification status.

In [ ]:
# saving projections as shapefiles, for each SSP and year
for ccode in ccode_dict:
    for n in range(1, n_scenario + 1):
        for x in range(start_year, end_year + year_int, year_int):
            elec_proj_name = ccode + '_elec_SSP' + str(n) + '_' + str(x)

            locals()[elec_proj_name].set_geometry('geometry', inplace=True, crs='EPSG:4326')
            locals()[elec_proj_name].to_file('4_output/elec_proj/' + ccode + '_elec_SSP' + str(n) + '_' + str(x) + '.gpkg', driver='GPKG')

In [ ]:
# preparing GeoDataFrames for rasterisation, i.e. combining all years in one GeoDataFrame for each SSP
for ccode in ccode_dict:
    for n in range(1, n_scenario + 1):
        elec_proj_all_name = ccode + '_elec_SSP' + str(n) # joined GeoDataFrame name
        
        for x in range(start_year, end_year, year_int):
            elec_proj_left_name = ccode + '_elec_SSP' + str(n) + '_' + str(x) # left GeoDataFrame name
            elec_proj_right_name = ccode + '_elec_SSP' + str(n) + '_' + str(x+5) # right GeoDataFrame name       
    
            locals()[elec_proj_right_name]['centroid'] = locals()[elec_proj_right_name].geometry.centroid
            locals()[elec_proj_right_name].set_geometry('centroid', inplace=True)
            
            # joining GeoDataFrames
            if x == 2025:
                locals()[elec_proj_all_name] = locals()[elec_proj_left_name].sjoin(locals()[elec_proj_right_name], how='left',
                                                                             predicate='contains')
            else:
                locals()[elec_proj_all_name] = locals()[elec_proj_all_name].sjoin(locals()[elec_proj_right_name], how='left',
                                                                             predicate='contains')
            # dropping repeated columns
            locals()[elec_proj_all_name].drop(columns=['index_right'], inplace=True)
            if x % 2 == 0:
                locals()[elec_proj_all_name].drop(columns=['geometry_right'], inplace=True)
        
        # dropping repeated columns
        locals()[elec_proj_all_name].drop(columns=['geometry_right', '2015_elec_right'], inplace=True)

In [ ]:
NAM_elec_SSP1

In [ ]:
# Defining a new function for rasterising GeoDataFrames
def gdf2rast(gdf, pixel_size, indicator, years, output_raster):
    '''
    This function rasterises the disaggregated SSP GeoDataFrames.
    Inputs:
        gdf (var): GeoDataFrame to rasterise
        pixel_size (int or float): pixel size in CRS units
        indicator (str): name of indicator whose data will be rasterised
        years (list): pathway years whose data will be rasterised
        output_raster (str): name of output raster file with .tif extension
    Returns:
        Raster file of disaggregated SSP data in multiple bands, each corresponding to an individual year.
    '''
    # GeoDataFrame to be rasterised
    gdf = gdf

    # output raster properties
    pixel_size = pixel_size  
    minx, miny, maxx, maxy = gdf.total_bounds  # retrieving the bounds of the GeoDataFrame
    width = int((maxx - minx) / pixel_size) # calculating cell width
    height = int((maxy - miny) / pixel_size) # calculating cell height
    transform = from_origin(minx, maxy, pixel_size, pixel_size)

    # defining the columns to be rasterised
    columns_to_rasterise = []
    for year in years:
        columns_to_rasterise.append(str(year) + '_' + indicator)

    # initialising an empty list to store the rasterised arrays
    rasters = []

    # rasterising each column and storing the result in the initialised list
    for column in columns_to_rasterise:
        shapes = ((geom, value) for geom, value in zip(gdf.geometry, gdf[column])) # retrieving cell geometry and indicator value
        raster = rasterize(shapes, out_shape=(height, width), transform=transform, fill=0, dtype='float32')
        rasters.append(raster)

    # defining the output raster file
    output_raster = output_raster

    # updating the profile and removing the NoData cells
    profile = {'driver': 'GTiff', # format
               'height': height,
               'width': width,
               'count': len(columns_to_rasterise), # band count
               'dtype': 'float32', # raster data type
               'crs': gdf.crs,
               'transform': transform,
               'nodata': 0} # removing nodata cells

    # saving the raster to a file with multiple bands, each corresponding to a year
    with rasterio.open(output_raster, 'w', **profile) as dst:
        for i, raster in enumerate(rasters, start=1):
            dst.write(raster, i) # writing raster values to band
            dst.set_band_description(i, columns_to_rasterise[i-1]) # naming the band according to indicator and year

    # printing confirmation message for the generated raster
    print(f"Raster with multiple bands saved to {output_raster}")

In [ ]:
# Rasterising avg. electricity demand, total electricity demand, MTF tiers & electrification status
for ccode in ccode_dict:
    for n in range(1, n_scenario + 1):
        elec_proj_all_name = ccode + '_elec_SSP' + str(n) # joined GeoDataFrame name

        # pixel size in degrees (e.g. 30 arc seconds, ~1 km), change as needed
        pixel_size = 30/3600 
        
        # indicator names
        ind_1 = 'perCapitaElec' 
        ind_2 = 'TotElec'
        ind_3 = 'MTF'
        ind_4 = 'elec'
        
        # modelled years, change as needed
        years = np.arange(2025,2055,5)

        # output file names
        out_1 = '4_output/perCapitaElec_rasters/' + ccode + '_SSP' + str(n) + '_' + ind_1 + '.tif' # output file name
        out_2 = '4_output/TotElec_rasters/' + ccode + '_SSP' + str(n) + '_' + ind_2 + '.tif'
        out_3 = '4_output/MTF_rasters/' + ccode + '_SSP' + str(n) + '_' + ind_3 + '.tif'
        out_4 = '4_output/ElecStat_rasters/' + ccode + '_SSP' + str(n) + '_' + ind_4 + '.tif'
        
        gdf2rast(locals()[elec_proj_all_name], pixel_size, ind_1, years, out_1) # rasterising per capita electricity consumption
        gdf2rast(locals()[elec_proj_all_name], pixel_size, ind_2, years, out_2) # rasterising total electricity consumption
        gdf2rast(locals()[elec_proj_all_name], pixel_size, ind_3, years, out_3) # rasterising MTF classes
        gdf2rast(locals()[elec_proj_all_name], pixel_size, ind_4, years, out_4) # rasterising electrification status

## End of residential electricity demand projection script

In [ ]:
# importing projections as GeoDataFrames, for each SSP and year
for ccode in ccode_dict:
    for n in range(1, n_scenario + 1):
        for x in range(start_year, end_year + year_int, year_int):
            elec_proj_name = ccode + '_elec_SSP' + str(n) + '_' + str(x)
            file_path = 'output/elec_proj/' + ccode + '_elec_SSP' + str(n) + '_' + str(x) + '/' + ccode + '_elec_SSP' + str(n) + '_' + str(x) + '.shp'

            locals()[elec_proj_name] = gpd.read_file(file_path)

            # classifying unelectrified cells as MTF Tier 0
            locals()[elec_proj_name].loc[locals()[elec_proj_name][str(x)+'_elec'] == False, str(x)+'_MTF'] = 0

In [ ]:
NAM_elec_SSP2_2035[NAM_elec_SSP2_2035['2035_elec']==False].groupby('2035_MTF')['2035_TotEl'].sum() / 10**6

In [ ]:
# Import global socioeconomic projections, according to the 5 SSPs
for n in range(1, n_scenario + 1):
    name_pop = 'SSP' + str(n) + '_POP_All'
    name_gdp = 'SSP' + str(n) + '_GDP_All'
    
    # importing GDP and POP files for each SSP (2025-2100, all countries)
    locals()[name_pop] = pd.read_excel('input/SSP_POP/SSP' + str(n) + '_POP_2025_2100_AllCountries.xlsx')
    locals()[name_gdp] = pd.read_excel('input/SSP_GDP/SSP' + str(n) + '_GDP_2025_2100_AllCountries.xlsx')

# importing Gini index projections (all SSPs, years, and GCAM regions)
Gini_GCAM = pd.read_excel('input/GCAM_Gini_SSP_1967_2100.xlsx', sheet_name='Gini_regions_filtered')
# importing GCAM region IDs (to map countries to GCAM regions)
GCAM_regID = pd.read_excel('input/GCAM_Gini_SSP_1967_2100.xlsx', sheet_name='iso_GCAM_regID')
# importing base year Gini indices (GCAM regions)
Gini_GCAM_base = pd.read_excel('input/GCAM_Gini_SSP_1967_2100.xlsx', sheet_name='Gini_regions_2015')
# importing base year Gini indices (all countries)
Gini_WB = pd.read_excel('input/WB_Gini_2015.xlsx', sheet_name='Gini_2015')

In [ ]:
for ccode in ccode_dict:
    pop_proj_name = ccode + '_pop_proj'
    gdp_proj_name = ccode + '_gdp_proj'
    inc_proj_name = ccode + '_inc_proj'
    
    # initialising population, GDP, and mean income (i.e. GDP per capita) dataframes
    locals()[pop_proj_name] = pd.DataFrame({'Year': np.arange(start_year, end_year + year_int, year_int)})
    locals()[gdp_proj_name] = pd.DataFrame({'Year': np.arange(start_year, end_year + year_int, year_int)})
    locals()[inc_proj_name] = pd.DataFrame({'Year': np.arange(start_year, end_year + year_int, year_int)})

    # populating the constructed dataframes
    for n in range(1, n_scenario + 1):
        name_pop = 'SSP' + str(n) + '_POP_All'
        name_gdp = 'SSP' + str(n) + '_GDP_All'
        
        # creating initial zero columns for each SSP
        col_name = 'SSP' + str(n)
        locals()[pop_proj_name][col_name] = 0
        locals()[gdp_proj_name][col_name] = 0
        
        for x in range(start_year, end_year + year_int, year_int):
            # original SSP datasets have population count in millions
            locals()[pop_proj_name].loc[locals()[pop_proj_name]['Year'] == x, [col_name]] = (locals()[name_pop].loc[locals()[name_pop]['Region'] == ccode_dict[ccode], str(x)].values[0] * 10 ** 6).round(0)
            # original SSP datasets have total GDP in billions
            locals()[gdp_proj_name].loc[locals()[gdp_proj_name]['Year'] == x, [col_name]] = locals()[name_gdp].loc[locals()[name_gdp]['Region'] == ccode_dict[ccode], str(x)].values[0] * 10 ** 9
            # calculating national average income level based on retrieved population and GDP projections
            locals()[inc_proj_name].loc[locals()[inc_proj_name]['Year'] == x, [col_name]] = \
            locals()[gdp_proj_name].loc[locals()[gdp_proj_name]['Year'] == x, [col_name]] / locals()[pop_proj_name].loc[locals()[pop_proj_name]['Year'] == x, [col_name]]

# displaying population dataframe as an example
NAM_pop_proj

In [ ]:
NAM_inc_proj

In [ ]:
for ccode in ccode_dict:
    elec_proj_name = ccode + '_elec_proj'
    tot_proj_name = ccode + '_totelec_proj'
    
    # initialising residential electricity demand dataframe
    locals()[elec_proj_name] = pd.DataFrame({'Year': np.arange(start_year, end_year + year_int, year_int)})
    locals()[tot_proj_name] = pd.DataFrame({'Year': np.arange(start_year, end_year + year_int, year_int)})

    # populating the constructed dataframe
    for n in range(1, n_scenario + 1):
        # creating initial zero columns for each SSP
        col_name = 'SSP' + str(n)
        locals()[elec_proj_name][col_name] = 0
        locals()[tot_proj_name][col_name] = 0
    
        for x in range(start_year, end_year + year_int, year_int):
            name_elec = ccode + '_elec_SSP' + str(n) + '_' + str(x)
            
            # populating each SSP column with country-specific average residential electricity demand
            locals()[elec_proj_name].loc[locals()[elec_proj_name]['Year'] == x, [col_name]] = (locals()[name_elec].loc[locals()[name_elec][str(x)+'_elec'] == True, str(x)+'_TotEl'].sum() / locals()[name_elec][str(x)+'_POP'].sum()).round(3)

            # populating each SSP column with country-specific total residential electricity demand
            locals()[tot_proj_name].loc[locals()[tot_proj_name]['Year'] == x, [col_name]] = (locals()[name_elec].loc[locals()[name_elec][str(x)+'_elec'] == True, str(x)+'_TotEl'].sum() / 10**6 ).round(3) # in GWh

# displaying the resulting DataFrame
NAM_elec_proj

In [ ]:
NAM_totelec_proj

In [ ]:
for ccode in ccode_dict:
    gini_proj_name = ccode + '_gini_proj'
    gcam_id_name = ccode + '_gcam_id'
    
    # initialising gini index dataframes
    locals()[gini_proj_name] = pd.DataFrame({'Year': np.arange(start_year, end_year + year_int, year_int)})

    # retrieving corresponding GCAM region ID
    locals()[gcam_id_name] = GCAM_regID.loc[GCAM_regID['iso'] == ccode.lower(), 'GCAM_region_ID'].values[0]

    # retrieving base year Gini indices for country and corresponding GCAM region
    cgini_base = Gini_WB.loc[Gini_WB['Country Code'] == ccode, 'Gini_2015'].values[0] # country
    rgini_base = Gini_GCAM_base.loc[Gini_GCAM_base['GCAM_region_ID'] == locals()[gcam_id_name], 'gini'].values[0] # GCAM region
    corr_fac = cgini_base / rgini_base # correction factor to be applied to regional Gini projections

    # populating the constructed dataframe
    for n in range(1, n_scenario + 1):        
        # creating initial zero columns for each SSP
        col_name = 'SSP' + str(n)
        locals()[gini_proj_name][col_name] = 0
        
        for x in range(start_year, end_year + year_int, year_int):
            # retrieving pojected Gini index for each year and SSP narrative,
            # then multiplying by correction factor to align GCAM trend with country starting point
            locals()[gini_proj_name].loc[locals()[gini_proj_name]['Year'] == x, [col_name]] = (Gini_GCAM.loc[(Gini_GCAM['GCAM_region_ID'] == locals()[gcam_id_name]) & (Gini_GCAM['year'] == x) & (Gini_GCAM['sce'] == col_name), 'gini'].values[0]) * corr_fac

# displaying one of the resulting dataframes
NAM_gini_proj

In [ ]:
for ccode in ccode_dict:
    access_proj_name = ccode + '_access_proj'
    
    # initialising residential electricity demand dataframe
    locals()[access_proj_name] = pd.DataFrame({'Year': np.arange(start_year, end_year + year_int, year_int)})

    # populating the constructed dataframe
    for n in range(1, n_scenario + 1):
        # creating initial zero columns for each SSP
        col_name = 'SSP' + str(n)
        locals()[access_proj_name][col_name] = 0
    
        for x in range(start_year, end_year + year_int, year_int):
            # retrieving projected electricity access rate for each scenario and year
            locals()[access_proj_name].loc[locals()[access_proj_name]['Year'] == x, [col_name]] = elec_access_proj.loc[(elec_access_proj['iso_code'] == ccode) & (elec_access_proj['ssp'] == n) & (elec_access_proj['year'] == x), 'electr'].values[0]

# displaying the resulting DataFrame
NAM_access_proj

In [ ]:
#locals()[elec_proj_name][str(x) + '_perCapitaElec'] = sigmoid(locals()[elec_proj_name][str(x) + '_INC'], *params)


for ccode in ccode_dict:
    pop_proj_name = ccode + '_pop_proj'
    inc_proj_name = ccode + '_inc_proj'
    access_proj_name = ccode + '_access_proj'
    elec_proj_single_name = ccode + '_elec_proj_single'
    tot_proj_single_name = ccode + '_totelec_proj_single'
    
    # initialising residential electricity demand dataframe, average and total, using single-point estimates
    locals()[elec_proj_single_name] = pd.DataFrame({'Year': np.arange(start_year, end_year + year_int, year_int)})
    locals()[tot_proj_single_name] = pd.DataFrame({'Year': np.arange(start_year, end_year + year_int, year_int)})

    # populating the constructed dataframes
    for n in range(1, n_scenario + 1):
        # creating initial zero columns for each SSP
        col_name = 'SSP' + str(n)
        locals()[elec_proj_single_name][col_name] = 0
        locals()[tot_proj_single_name][col_name] = 0
    
        for x in range(start_year, end_year + year_int, year_int):
            # calculating average demand based on mean income only, for each scenario and year
            locals()[elec_proj_single_name].loc[locals()[elec_proj_single_name]['Year'] == x, [col_name]] = sigmoid(locals()[inc_proj_name].loc[locals()[inc_proj_name]['Year'] == x, [col_name]], *params)
            # calculating total demand based on population and access rate, for each scenario and year
            locals()[tot_proj_single_name].loc[locals()[tot_proj_single_name]['Year'] == x, [col_name]] = locals()[elec_proj_single_name].loc[locals()[elec_proj_single_name]['Year'] == x, [col_name]] * locals()[pop_proj_name].loc[locals()[pop_proj_name]['Year'] == x, [col_name]] * (locals()[access_proj_name].loc[locals()[access_proj_name]['Year'] == x, [col_name]]/100) / 10**6
            

# displaying the resulting DataFrame
NAM_elec_proj_single

In [ ]:
NAM_totelec_proj_single

In [ ]:
for ccode in ccode_dict:
    pop_proj_name = ccode + '_pop_proj'
    inc_proj_name = ccode + '_inc_proj'
    elec_proj_name = ccode + '_elec_proj'
    gini_proj_name = ccode + '_gini_proj'
    access_proj_name = ccode + '_access_proj'
    
    # plotting summary stats for population, income level & electricity demand development by SSP
    fig, axes = plt.subplots(2, 2, sharex=True, figsize=(8,8)) # creating 4 sub-plots for the 4 indicators
    ax1, ax2 = axes[0]
    ax3, ax4 = axes[1]
    
    # 1st sub-plot: total population development
    locals()[pop_proj_name].plot(x='Year', kind='line', ax=ax1, ylabel='Population [capita]', legend=False)
    
    # 2nd sub-plot: electricity access rate development
    ls = ['-', '-', '-', '-.', '-.']
    for n in range(1, n_scenario + 1):
        ax2.plot(locals()[access_proj_name]['Year'], locals()[access_proj_name]['SSP'+str(n)], linestyle=ls[n-1])
    ax2.set_xlabel('Year')
    ax2.set_ylabel('Electricity access rate [%]')    
    #locals()[access_proj_name].plot(x='Year', kind='line', ax=ax2, ylabel='Electricity access rate [%]', linestyle='--')
    
    # 3rd sub-plot: average income level development
    locals()[inc_proj_name].plot(x='Year', kind='line', ax=ax3, ylabel='Income level, PPP [constant 2017 intl. USD/cap]', legend=False)
    
    # 4th sub-plot: Gini index development
    locals()[gini_proj_name].plot(x='Year', kind='line', ax=ax4, ylabel='Gini index [-]', legend=False)
    # 3rd sub-plot: average residential electricity demand development
    #locals()[elec_proj_name].plot(x='Year', kind='line', ax=ax3, ylabel='Residential electricity demand [kWh/cap/a]', linestyle='--')
    
    # pop sub-plot
    ax1.ticklabel_format(style='sci', axis='y', scilimits=(6, 6))
    ax1.grid(which='both', alpha=0.25)
    # access sub-plot
    ax2.grid(which='both', alpha=0.25)
    # inc sub-plot
    ax3.ticklabel_format(style='sci', axis='y', scilimits=(3, 3))
    ax3.grid(which='both', alpha=0.25)
    # gini sub-plot
    ax4.grid(which='both', alpha=0.25)
    
    #ax3.set_ylim(ymin=350)
    #ax3.grid(which='both', alpha=0.25)

    # Collect handles and labels from one of the plots (e.g.,ax2,sinceyoumanuallyset SSP labels there)
    handles, labels = ax1.get_legend_handles_labels()
   
 
    # Place a single legend for all subplots
    fig.legend(handles, labels, loc='upper center', ncol=n_scenario, bbox_to_anchor=(0.5, 1.0))
    
    plt.tight_layout(rect=[0, 0, 1, 0.98])  # leave space for legend
    
    plt.rcParams['font.family'] = 'sans-serif' # font type
    plt.rcParams['font.size'] = 8 # font size
    plt.savefig('output/Figures/' + ccode + '_summary_stats_pop_acc_inc_gini.png', dpi=600) # saving output figure: file name and resolution
    plt.show()

In [ ]:
from matplotlib import cm
from matplotlib.colors import ListedColormap
import numpy as np

viridis = cm.get_cmap('viridis', 5)
newcolors = viridis(np.linspace(0, 1, 5))
grey = np.array([173/256, 173/256, 173/256, 1])
newcolors = np.vstack([newcolors, grey])
newcmp = ListedColormap(newcolors)
newcmp

In [ ]:
# plotting summary stats for population & total electricity demand by MTF tiers
for ccode in ccode_dict:
    # setting up required DataFrames for plotting
    pop_by_MTF = pd.DataFrame({'MTF': np.arange(0, 6)}) # empty population DataFrame, columns correspond to MTF tiers
    elec_by_MTF = pd.DataFrame({'MTF': np.arange(0, 6)}) # empty electricity demand DataFrame, columns correspond to MTF tiers
    year = str(2050) # plotted year, change as needed
    
    for n in range(1, n_scenario + 1):
        elec_name = ccode + '_elec_SSP' + str(n) + '_' + year
        
        # population in millions, summed up and grouped by MTF tiers
        pop_by_MTF['SSP'+str(n)] = locals()[elec_name].groupby(year+'_MTF')[year+'_POP'].sum().reindex(np.arange(0, 6), fill_value=0).values / 10 ** 6
        # total residential electricity demand in GWh, summed up and grouped by MTF tiers
        elec_by_MTF['SSP'+str(n)] = locals()[elec_name].groupby(year+'_MTF')[year+'_TotEl'].sum().reindex(np.arange(0, 6), fill_value=0).values / 10 ** 6
    
    # transposing DataFrames for proper plotting
    pop_by_MTF = pop_by_MTF.iloc[:,1:].transpose()
    pop_by_MTF.columns = ['unelectrified', 'Tier 1', 'Tier 2', 'Tier 3', 'Tier 4', 'Tier 5'] # renaming columns to full name instead of a single number
    pop_by_MTF = pop_by_MTF[['Tier 1', 'Tier 2', 'Tier 3', 'Tier 4', 'Tier 5', 'unelectrified']] # rearrange columns
    print(pop_by_MTF) # displaying resulting table
    
    elec_by_MTF = elec_by_MTF.iloc[:,1:].transpose()
    elec_by_MTF.columns = ['unelectrified', 'Tier 1', 'Tier 2', 'Tier 3', 'Tier 4', 'Tier 5'] # renaming columns to full name instead of a single number
    elec_by_MTF = elec_by_MTF[['Tier 1', 'Tier 2', 'Tier 3', 'Tier 4', 'Tier 5', 'unelectrified']] # rearrange columns
    print(elec_by_MTF) # displaying resulting table
    
    # creating 2 sub-plots for the 2 indicators
    fig, (ax1, ax2) = plt.subplots(1,2, sharex=True, figsize=(10,5)) 
    
    # 1st sub-plot: total population in 2050 across different SSPs, broken down by MTF tier
    pop_by_MTF.plot(kind='bar', stacked=True, cmap=newcmp, ylabel='Population [mil]', ax=ax1)
    ax1.get_legend().remove() # removing legend of 1st sub-plot, only one legend is used for both plots to avoid redundancy
    
    # 2nd sub-plot: total residential electricity demand in 2050 across different SSPs, broken down by MTF tier
    elec_by_MTF.plot(kind='bar', stacked=True, cmap=newcmp, ylabel='Residential electricity demand [GWh]', ax=ax2)
    ax2.legend(bbox_to_anchor = (1.3, 0.5), loc='center right', reverse=True) # setting legend properties
    
    plt.rcParams['font.family'] = 'sans-serif' # font type
    plt.rcParams['font.size'] = 7 # font size
    plt.savefig('output/Figures/' + ccode + '_summary_stats_byMTF_' + year + '.png', dpi=300) # saving output figure: file name and resolution
    plt.show()

In [ ]:
# plotting summary stats for population & total electricity demand by MTF tiers
for ccode in ccode_dict:
    
    #year = str(2050) # plotted year, change as needed
    
    for n in range(1, n_scenario + 1):
        # setting up required DataFrames for plotting
        pop_by_MTF = pd.DataFrame({'MTF': np.arange(0, 6)}) # empty population DataFrame, columns correspond to MTF tiers
        elec_by_MTF = pd.DataFrame({'MTF': np.arange(0, 6)}) # empty electricity demand DataFrame, columns correspond to MTF tiers
        for x in range(start_year, end_year + year_int, year_int):
            elec_name = ccode + '_elec_SSP' + str(n) + '_' + str(x)
            
            # population in millions, summed up and grouped by MTF tiers
            pop_by_MTF[str(x)] = locals()[elec_name].groupby(str(x)+'_MTF')[str(x)+'_POP'].sum().reindex(np.arange(0, 6), fill_value=0).values / 10 ** 6
            # total residential electricity demand in GWh, summed up and grouped by MTF tiers
            elec_by_MTF[str(x)] = locals()[elec_name].groupby(str(x)+'_MTF')[str(x)+'_TotEl'].sum().reindex(np.arange(0, 6), fill_value=0).values / 10 ** 6
    
        # transposing DataFrames for proper plotting
        pop_by_MTF = pop_by_MTF.iloc[:,1:].transpose()
        pop_by_MTF.columns = ['unelectrified', 'Tier 1', 'Tier 2', 'Tier 3', 'Tier 4', 'Tier 5'] # renaming columns to full name instead of a single number
        pop_by_MTF = pop_by_MTF[['Tier 1', 'Tier 2', 'Tier 3', 'Tier 4', 'Tier 5', 'unelectrified']] # rearrange columns
        print(pop_by_MTF) # displaying resulting table
        
        elec_by_MTF = elec_by_MTF.iloc[:,1:].transpose()
        elec_by_MTF.columns = ['unelectrified', 'Tier 1', 'Tier 2', 'Tier 3', 'Tier 4', 'Tier 5'] # renaming columns to full name instead of a single number
        elec_by_MTF = elec_by_MTF[['Tier 1', 'Tier 2', 'Tier 3', 'Tier 4', 'Tier 5', 'unelectrified']] # rearrange columns
        print(elec_by_MTF) # displaying resulting table
    
        # creating 2 sub-plots for the 2 indicators
        fig, (ax1, ax2) = plt.subplots(1,2, sharex=True, figsize=(10,5)) 
        
        # 1st sub-plot: total population in 2050 across different SSPs, broken down by MTF tier
        pop_by_MTF.plot(kind='bar', stacked=True, cmap=newcmp, ylabel='Population [mil]', ax=ax1)
        ax1.get_legend().remove() # removing legend of 1st sub-plot, only one legend is used for both plots to avoid redundancy
        ax1.set_ylim(top=4)
        ax1.grid(which='major', axis='y', alpha=0.25)
        
        # 2nd sub-plot: total residential electricity demand in 2050 across different SSPs, broken down by MTF tier
        elec_by_MTF.plot(kind='bar', stacked=True, cmap=newcmp, ylabel='Residential electricity demand [GWh]', ax=ax2)
        ax2.legend(bbox_to_anchor = (1.3, 0.5), loc='center right', reverse=True) # setting legend properties
        ax2.set_ylim(top=3500)
        ax2.grid(which='major', axis='y', alpha=0.25)
        
        plt.rcParams['font.family'] = 'sans-serif' # font type
        plt.rcParams['font.size'] = 7 # font size
        plt.savefig('output/Figures/' + ccode + '_SSP' + str(n) + '_summary_stats_byMTF.png', dpi=300) # saving output figure: file name and resolution
        plt.show()

In [ ]:
import matplotlib.pyplot as plt
for ccode in ccode_dict:
    tot_elec_proj = locals()[ccode + '_totelec_proj']
    tot_elec_proj_single = locals()[ccode + '_totelec_proj_single']

    fig, axes = plt.subplots(3, 3, sharex=False, sharey=False, figsize=(12,12))
    #df.plot('x',y=['y_one','y_two'])
    axes[0,0].plot(tot_elec_proj['Year'], tot_elec_proj['SSP1'].values/1000, 'k-', label='high-resolution estimate')
    axes[0,0].plot(tot_elec_proj_single['Year'], tot_elec_proj_single['SSP1'].values/1000, 'r-', label='single-point estimate')
    axes[0,0].set_ylabel('Residential electricity demand [TWh]')
    axes[0,0].grid(which='both', axis='y', alpha=0.25)
    axes[0,0].legend(loc='upper left')
    #axes[0,0].set_title('SSP1')
    axes[0,0].set_ylim(bottom=0, top=3.750)
    
    axes[1,0].plot(tot_elec_proj['Year'], tot_elec_proj['SSP2'].values/1000, 'k-', label='high-resolution estimate')
    axes[1,0].plot(tot_elec_proj_single['Year'], tot_elec_proj_single['SSP2'].values/1000, 'r-', label='single-point estimate')
    axes[1,0].set_ylabel('Residential electricity demand [TWh]')
    axes[1,0].grid(which='both', axis='y', alpha=0.25)
    #axes[1,0].set_title('SSP2')
    axes[1,0].set_ylim(bottom=0, top=3.750)
    
    axes[2,0].plot(tot_elec_proj['Year'], tot_elec_proj['SSP3'].values/1000, 'k-', label='high-resolution estimate')
    axes[2,0].plot(tot_elec_proj_single['Year'], tot_elec_proj_single['SSP3'].values/1000, 'r-', label='single-point estimate')
    axes[2,0].set_ylabel('Residential electricity demand [TWh]')
    axes[2,0].grid(which='both', axis='y', alpha=0.25)
    #axes[2,0].set_title('SSP3')
    axes[2,0].set_ylim(bottom=0, top=3.750)

    #plt.show()
    for n in [1, 2, 3]:
        # setting up required DataFrames for plotting
        pop_by_MTF = pd.DataFrame({'MTF': np.arange(0, 6)}) # empty population DataFrame, columns correspond to MTF tiers
        elec_by_MTF = pd.DataFrame({'MTF': np.arange(0, 6)}) # empty electricity demand DataFrame, columns correspond to MTF tiers
        for x in range(start_year, end_year + year_int, year_int):
            elec_name = ccode + '_elec_SSP' + str(n) + '_' + str(x)
            
            # population in millions, summed up and grouped by MTF tiers
            pop_by_MTF[str(x)] = locals()[elec_name].groupby(str(x)+'_MTF')[str(x)+'_POP'].sum().reindex(np.arange(0, 6), fill_value=0).values / 10 ** 6
            # total residential electricity demand in TWh, summed up and grouped by MTF tiers
            elec_by_MTF[str(x)] = locals()[elec_name].groupby(str(x)+'_MTF')[str(x)+'_TotEl'].sum().reindex(np.arange(0, 6), fill_value=0).values / 10 ** 9
    
        # transposing DataFrames for proper plotting
        pop_by_MTF = pop_by_MTF.iloc[:,1:].transpose()
        pop_by_MTF.columns = ['unelectrified', 'Tier 1', 'Tier 2', 'Tier 3', 'Tier 4', 'Tier 5'] # renaming columns to full name instead of a single number
        pop_by_MTF = pop_by_MTF[['Tier 1', 'Tier 2', 'Tier 3', 'Tier 4', 'Tier 5', 'unelectrified']] # rearrange columns
        #print(pop_by_MTF) # displaying resulting table
        
        elec_by_MTF = elec_by_MTF.iloc[:,1:].transpose()
        elec_by_MTF.columns = ['unelectrified', 'Tier 1', 'Tier 2', 'Tier 3', 'Tier 4', 'Tier 5'] # renaming columns to full name instead of a single number
        elec_by_MTF = elec_by_MTF[['Tier 1', 'Tier 2', 'Tier 3', 'Tier 4', 'Tier 5', 'unelectrified']] # rearrange columns
        #print(elec_by_MTF) # displaying resulting table
    
        # creating 2 sub-plots for the 2 indicators
        #fig, (ax1, ax2) = plt.subplots(1,2, sharex=True, figsize=(10,5)) 
        
        # 1st sub-plot: total population in 2050 across different SSPs, broken down by MTF tier
        pop_by_MTF.plot(kind='bar', stacked=True, cmap=newcmp, ylabel='Population [mil]', ax=axes[n-1,2], rot=0)
        axes[n-1,2].get_legend().remove() # removing legend of 1st sub-plot, only one legend is used for both plots to avoid redundancy
        axes[n-1,2].set_ylim(top=4)
        axes[n-1,2].grid(which='major', axis='y', alpha=0.25)
        
        # 2nd sub-plot: total residential electricity demand in 2050 across different SSPs, broken down by MTF tier
        elec_by_MTF.plot(kind='bar', stacked=True, cmap=newcmp, ylabel='Residential electricity demand [TWh]', ax=axes[n-1,1], rot=0)
        #ax2.legend(bbox_to_anchor = (1.3, 0.5), loc='center right', reverse=True) # setting legend properties
        if n != 1:
            axes[n-1,1].get_legend().remove()
        else:
            axes[n-1,1].legend(loc='upper left', reverse=True)
        
        axes[n-1,1].set_ylim(top=3.750)
        axes[n-1,1].grid(which='major', axis='y', alpha=0.25)
        axes[n-1,1].set_title('SSP'+str(n), fontweight='bold')
        
    
    plt.rcParams['font.family'] = 'sans-serif' # font type
    plt.rcParams['font.size'] = 8 # font size
    plt.savefig('output/Figures/' + ccode + '_SSP123_summary_stats_byMTF.png', dpi=600) # saving output figure: file name and resolution
    plt.show()
        

In [ ]:
# plotting summary stats for population & total electricity demand by MTF tiers
for ccode in ccode_dict:
    
    #year = str(2050) # plotted year, change as needed
    
    for n in range(1, n_scenario + 1):
        # setting up required DataFrames for plotting
        pop_by_MTF = pd.DataFrame({'MTF': np.arange(0, 6)}) # empty population DataFrame, columns correspond to MTF tiers
        elec_by_MTF = pd.DataFrame({'MTF': np.arange(0, 6)}) # empty electricity demand DataFrame, columns correspond to MTF tiers
        for x in range(start_year, end_year + year_int, year_int):
            elec_name = ccode + '_elec_SSP' + str(n) + '_' + str(x)
            
            # population in millions, summed up and grouped by MTF tiers
            pop_by_MTF[str(x)] = locals()[elec_name].groupby(str(x)+'_MTF')[str(x)+'_POP'].sum().reindex(np.arange(0, 6), fill_value=0).values / 10 ** 6
            # total residential electricity demand in GWh, summed up and grouped by MTF tiers
            elec_by_MTF[str(x)] = locals()[elec_name].groupby(str(x)+'_MTF')[str(x)+'_TotEl'].sum().reindex(np.arange(0, 6), fill_value=0).values / 10 ** 6
    
        # transposing DataFrames for proper plotting
        pop_by_MTF = pop_by_MTF.iloc[:,1:].transpose()
        pop_by_MTF.columns = ['unelectrified', 'Tier 1', 'Tier 2', 'Tier 3', 'Tier 4', 'Tier 5'] # renaming columns to full name instead of a single number
        pop_by_MTF = pop_by_MTF[['Tier 1', 'Tier 2', 'Tier 3', 'Tier 4', 'Tier 5', 'unelectrified']] # rearrange columns
        print(pop_by_MTF) # displaying resulting table
        
        elec_by_MTF = elec_by_MTF.iloc[:,1:].transpose()
        elec_by_MTF.columns = ['unelectrified', 'Tier 1', 'Tier 2', 'Tier 3', 'Tier 4', 'Tier 5'] # renaming columns to full name instead of a single number
        elec_by_MTF = elec_by_MTF[['Tier 1', 'Tier 2', 'Tier 3', 'Tier 4', 'Tier 5', 'unelectrified']] # rearrange columns
        print(elec_by_MTF) # displaying resulting table
    
        # creating 2 sub-plots for the 2 indicators
        fig, (ax1, ax2) = plt.subplots(1,2, sharex=True, figsize=(10,5)) 
        
        # 1st sub-plot: total population in 2050 across different SSPs, broken down by MTF tier
        pop_by_MTF.plot(kind='bar', stacked=True, cmap=newcmp, ylabel='Population [mil]', ax=ax1)
        ax1.get_legend().remove() # removing legend of 1st sub-plot, only one legend is used for both plots to avoid redundancy
        ax1.set_ylim(top=4)
        ax1.grid(which='major', axis='y', alpha=0.25)
        
        # 2nd sub-plot: total residential electricity demand in 2050 across different SSPs, broken down by MTF tier
        elec_by_MTF.plot(kind='bar', stacked=True, cmap=newcmp, ylabel='Residential electricity demand [GWh]', ax=ax2)
        ax2.legend(bbox_to_anchor = (1.3, 0.5), loc='center right', reverse=True) # setting legend properties
        ax2.set_ylim(top=3500)
        ax2.grid(which='major', axis='y', alpha=0.25)
        
        plt.rcParams['font.family'] = 'sans-serif' # font type
        plt.rcParams['font.size'] = 7 # font size
        #plt.savefig('output/Figures/' + ccode + '_SSP' + str(n) + '_summary_stats_byMTF.png', dpi=300) # saving output figure: file name and resolution
        plt.show()

In [ ]:
import folium
import branca.colormap as cm
from branca.colormap import linear
from branca.element import Element

ind = 'perCapitaElec' # specify indicator to be plotted

# Prepare the GeoDataFrame
gdf = res_elec_SSP1_all.drop(columns=['centroid']).set_geometry('geometry')

# Define colormap intervals and bounds
#bounds = [1, 2, 3, 4, 5, 6]  # 5 intervals for 5 tiers
bounds = mtf_bins # intervals for per capita electricity demand

# Use the correct attribute name for the colormap
#colormap = cm.StepColormap(['#440154', '#3b518b', '#21908d', '#5cc863', '#fde725'], vmin = np.log10(bounds[0]), vmax = np.log10(bounds[-1]), index=np.log10(bounds))
colormap = cm.StepColormap(['#440154', '#3b518b', '#21908d', '#5cc863', '#fde725'], vmin = bounds[0], vmax = bounds[-2]+50, index=bounds[:-1], max_labels=6)
#colormap.scale(vmin = bounds[0], vmax = bounds[-1], max_labels = 10)
#colormap = cm.viridis
#colormap = colormap.to_step(index=bounds, method='left')
colormap.caption = 'Residential electricity demand in 2050 [kWh/cap/a]'

# Center the map
m = folium.Map(location=[-23, 17], zoom_start=7, tiles="cartodbpositron")

# Fix: Ensure the GeoJson properties include '2050_MTF' by resetting the index
#gdf = gdf.reset_index(drop=True)

# Fix: Pass the correct field name as it appears in the GeoJSON properties
# Sometimes, after processing, the column name may be changed or lost.
# To debug, let's check the columns:
# print(gdf.columns)

tooltip = folium.GeoJsonTooltip(
    fields=['POP', 'INC', 'perCapitaElec'],
    aliases=['Population count:', 'Income level [USD/capita]:', 'Residential electricity demand [kWh/capita]:'],
    localize=True,
    sticky=False,
    labels=True,
    style="""
        background-color: #F0EFEF;
        border: 2px solid black;
        border-radius: 3px;
        box-shadow: 3px;
    """,
    max_width=800,
)

# Add polygons colored by MTF tier
folium.GeoJson(
    gdf.tail(1000),
    style_function=lambda feature: {
        'fillColor': colormap(feature['properties']['perCapitaElec']),
        'color': 'black',
        'weight': 0.2,
        'fillOpacity': 0.7,
    },
    name='Electricity demand',
    tooltip=tooltip
).add_to(m)

colormap.add_to(m)
#e = Element("""
#  var ticks = document.querySelectorAll('div.legend g.tick text')
#  for(var i = 0; i < ticks.length; i++) {
#    var value = parseFloat(ticks[i].textContent.replace(',', ''))
#    var newvalue = Math.pow(10.0, value).toFixed(0).toString()
#    ticks[i].textContent = newvalue
#  }
#""")
#colormap = colo.color_scale
#html = colormap.get_root()
#html.script.get_root().render()
#html.script.add_child(e)

m
#m.save('map_perCapitaElec.html')

In [ ]:
help(folium.plugins.TimeSliderChoropleth)

In [ ]:
import folium
import numpy as np
import pandas as pd
from branca.element import Element

# Ensure 'index' is a column in the DataFrame
gdf = res_elec_SSP1_2050.drop(columns=['centroid']).set_geometry('geometry').head(1000).reset_index()
#gdf['2050_perCapitaElec'] = np.log10(gdf['2050_perCapitaElec'])
bounds = mtf_bins
m = folium.Map(location=[-23, 17], zoom_start=7)

c = folium.Choropleth(
    geo_data=gdf,
    name="choropleth",
    data=gdf,
    columns=['index', '2050_perCapitaElec'],
    key_on='feature.properties.index',
    fill_color="viridis",
    scheme='UserDefined',
    classification_kwds={'bins': bounds},
    categorical=True,
    #bins=bounds,
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name='Residential electricity demand in 2050 [kWh/cap/a]',
)
#c.color_scale.scale(vmin=bounds[0], vmax=bounds[-2]+200, max_labels=4)
c.add_to(m)

#e = Element("""
#  var ticks = document.querySelectorAll('div.legend g.tick text')
#  for(var i = 0; i < ticks.length; i++) {
#    var value = parseFloat(ticks[i].textContent.replace(',', ''))
#    var newvalue = Math.pow(10.0, value).toFixed(0).toString()
#    ticks[i].textContent = newvalue
#  }
#""")
#colormap = c.color_scale
#html = colormap.get_root()
#html.script.get_root().render()
#html.script.add_child(e)
m

In [ ]:
!pip install leafmap

In [ ]:
import matplotlib.cm as cm

fig, ax = plt.subplots()

set_map = cm.viridis
set_bins=[15, 73, 250, 600, 7500]
set_bounds=[0.9, 15, 73, 250, 600]
set_norm = mpl.colors.BoundaryNorm(set_bounds, set_map.N, extend='max')
set_sm = plt.cm.ScalarMappable(cmap=set_map, norm=set_norm)
set_sm.set_array([])

fig.colorbar(set_sm, ax=ax, fraction=0.04, pad=0.01, boundaries=set_bounds, ticks=set_bounds,
             label='Residential electricity demand [kWh/cap/a]', orientation='horizontal')

mpl.patches.BoxStyle('round', pad=0.3)
fig.patch.set_facecolor('white')
fig.patch.set_alpha(0.8)
fig.patch.set_edgecolor('#575757')
fig.patch.set_linewidth(2.5)
fig.patch.set_capstyle('round')
ax.remove()
#plt.figure(facecolor='white')
plt.rcParams['font.family'] = 'sans-serif'
plt.savefig('plot_onlycbar_tight.png', bbox_inches='tight')
plt.show()


In [ ]:
import leafmap.foliumap as leafmap
import matplotlib.cm as cm

# Create a map centered at a specific location
m = leafmap.Map(center=[-23, 17], zoom=10)  # replace with your coordinates

gdf = res_elec_SSP1_2050.drop(columns=['centroid']).set_geometry('geometry')
bounds = mtf_bins[1:]

#m.zoom_to_gdf(gdf)

m.add_data(
    data=gdf,
    column='2050_perCapitaElec',
    layer_name='Electricity demand',
    cmap='viridis',
    scheme='UserDefined',
    classification_kwds={'bins': bounds},
    add_legend=False,
    #legend_title='Residential electricity demand in 2050 [kWh/cap/a]',
    #legend_kwds={"fmt": "{:.0f}"},
    #labels=[int(x) for x in mtf_bins[:-1]],
    stroke=True,
    color='#878787',
    opacity=0.9,
    weight=0.5,
    fields=['2050_POP', '2050_INC', '2050_perCapitaElec'],
    #style_function=lambda feature: {'weight': 0}
    #draggable=False
    #edgecolor='none'
    #style_function=lambda feature: {'stroke':False, 'fillOpacity':0.7}
)
#m.zoom_to_gdf(gdf)
m.add_image('plot_onlycbar_tight.png', position='bottomright')
#m.add_colorbar(['#440154', '#3b518b', '#21908d', '#5cc863', '#fde725'],
#               vmin=0.9,
#               vmax=650,
#               index=[15, 73, 250, 600, 650],
#               categorical=False)

#m.add_colormap(width=4.0, height=0.3, vmin=0.9, vmax=7500, palette=None, vis_params=None, cmap='viridis', discrete=True, label=None, label_size=12, label_weight='normal', tick_size=10, bg_color='white', orientation='horizontal', dpi='figure', transparent=True, position=(0, 0))

#m.add_data(
#    data=gdf_fixed,
#    column='2050_MTF',
#    layer_name='MTF',
#    cmap='viridis',
    #scheme='UserDefined',
    #classification_kwds={'bins': bounds},
#    legend=True,
#    legend_title='MTF tiers',
#    labels=[1, 2, 3, 4, 5],
#    opacity=0.7,
#    fields=['2050_POP', '2050_INC', '2050_perCapitaElec'],
    #edgecolor='none'
    #style_function=lambda feature: {'stroke':False, 'fillOpacity':0.7}
#)

m.to_html('map_2050_perCapitaElec.html')
#m

In [ ]:
res_elec_SSP1_2025

In [ ]:
for n in range(1,6):
    ssp_list = []
    ssp_concat = 'res_elec_SSP' + str(n) + '_all'
    for x in range(2025,2055,5):
        gdf_name = 'res_elec_SSP' + str(n) + '_' + str(x)
        locals()[gdf_name]['Year'] = x
        locals()[gdf_name].rename(columns={str(x) + '_POP': 'POP',
                                           str(x) + '_INC': 'INC',
                                           str(x) + '_perCapitaElec': 'perCapitaElec',
                                           str(x) + '_TotElec': 'TotElec',
                                           str(x) + '_MTF': 'MTF',
                                           str(x) + '_elec': 'curr_elec'},
                                 inplace=True)
        ssp_list.append(locals()[gdf_name])
    locals()[ssp_concat] = gpd.GeoDataFrame(pd.concat(ssp_list, ignore_index=True), crs=ssp_list[0].crs)

In [ ]:
res_elec_SSP3_2045

In [ ]:
res_elec_SSP3_all

In [ ]:
from leafmap import leafmap
#import leafmap.foliumap as leafmap
import matplotlib.cm as cm

# Prepare the GeoDataFrame for the time slider
# The 'Year' column must be present and of type int or str
gdf = res_elec_SSP1_all.drop(columns=['centroid']).set_geometry('geometry')

#gdf['Year'] = gdf['Year'].astype(str)  # TimeSliderChoropleth expects string or int

bounds = mtf_bins[1:]

# Create a map centered at a specific location
m = leafmap.Map(center=[-23, 17], zoom=10)

# Add the time slider layer
m.add_gdf_time_slider(gdf[['geometry', 'perCapitaElec', 'Year']], time_columns=['Year'], time_interval=5)

m.add_image('plot_onlycbar_tight.png', position='bottomright')

m.to_html('map_perCapitaElec_SSP1_timeslider.html')
#m

In [ ]:
help(leafmap.Map.add_gdf_time_slider)

In [ ]:
set_sm

In [ ]:
import branca.colormap as cm
m = folium.Map(tiles="cartodbpositron")
'''
set_map = cm.viridis
set_bins=[15, 73, 250, 600, 7500]
set_bounds=[0.9, 15, 73, 250, 600]
set_norm = mpl.colors.BoundaryNorm(set_bounds, set_map.N, extend='max')
set_sm = plt.cm.ScalarMappable(cmap=set_map, norm=set_norm)
set_sm.set_array([])
'''
colormap = cm.linear.viridis.to_step(n=5, method='log', index=[0.9, 15, 73, 250, 600, 700])
colormap.caption = "A colormap caption"
m.add_child(colormap)

m

In [ ]:
import leafmap.leafmap as leafmap

url = "https://github.com/opengeos/datasets/releases/download/us/zillow_home_value_index_by_county_ca.geojson"
data = gpd.read_file(url)
data.head(2)

legend_dict = {
    "[ 0,  200000]": "#e6f3ff",  # Very light blue
    "( 200000,  400000]": "#deebf7",  # Light blue
    "( 400000,  600000]": "#9ecae1",  # Medium blue
    "( 600000,  800000]": "#4292c6",  # Medium-dark blue
    "( 800000, 1000000]": "#2171b5",  # Dark blue
    "( 1000000, 2000000]": "#084594",  # Very dark blue
    "Nodata": "#f0f0f0",  # Light gray
}

gdf = leafmap.color_code_dataframe(data, legend_dict=legend_dict)
gdf.head(2)

m = leafmap.Map()

m.add_gdf_time_slider(gdf, time_interval=0.05, zoom_to_layer=True)
m.add_legend(title="Median Home Value", legend_dict=legend_dict)

m

In [ ]:
import folium
from folium.plugins import TimestampedGeoJson
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np

# Use res_elec_SSP1_all GeoDataFrame
gdf = res_elec_SSP1_all.copy()
gdf['Year'] = gdf['Year'].astype(str)

# Create a colormap for perCapitaElec
vmin = gdf['perCapitaElec'].min()
vmax = gdf['perCapitaElec'].max()
cmap = plt.cm.viridis
norm = mpl.colors.Normalize(vmin=vmin, vmax=vmax)

def get_color(val):
    rgba = cmap(norm(val))
    return mpl.colors.rgb2hex(rgba)

# Create features for GeoJSON (polygons, colored by perCapitaElec)
features = []
for _, row in gdf.iterrows():
    color = get_color(row['perCapitaElec'])
    feature = {
        'type': 'Feature',
        'geometry': row['geometry'].__geo_interface__,
        'properties': {
            'time': row['Year'],
            'style': {
                'color': color,
                'fillColor': color,
                'weight': 1,
                'fillOpacity': 0.7
            },
            'perCapitaElec': row['perCapitaElec']
        }
    }
    features.append(feature)

geojson = {
    'type': 'FeatureCollection',
    'features': features
}

# Center map on mean coordinates
if not gdf.empty:
    centroid = gdf.geometry.unary_union.centroid
    map_center = [centroid.y, centroid.x]
else:
    map_center = [0, 0]

m = folium.Map(location=map_center, zoom_start=5)

# Add time slider
TimestampedGeoJson(
    geojson,
    transition_time=200,
    period='P1Y',  # yearly interval
    add_last_point=True,
    auto_play=False,
    loop=False,
    max_speed=1,
    loop_button=True,
    date_options='YYYY',
    time_slider_drag_update=True,
    style=lambda feat: feat['properties']['style']
).add_to(m)

# Add a colormap legend
from branca.colormap import linear
colormap = linear.Viridis_09.scale(vmin, vmax)
colormap.caption = 'perCapitaElec'
colormap.add_to(m)

m

In [ ]:
viridis = cm.get_cmap('viridis', 5)
newcolors = viridis(np.linspace(0, 1, 5))
grey = np.array([173/256, 173/256, 173/256, 1])
newcolors = np.vstack([newcolors, grey])
newcmp = ListedColormap(newcolors)
newcmp

In [ ]:
NAM_elec_SSP1_2025.head()

In [ ]:
import folium
import geopandas as gpd
import matplotlib
import matplotlib.cm as cm
import numpy as np

for ccode in ccode_dict:
    # Get centroid for initial map location using the first scenario's gdf
    gdf0 = locals()[ccode + '_elec_SSP1_2025'].copy()
    gdf0 = gdf0.to_crs(epsg=4326)
    centroid = gdf0.geometry.unary_union.centroid
    m = folium.Map(location=[centroid.y, centroid.x], zoom_start=7)
    
    # Prepare colormap for MTF (1-5, plus grey for unelectrified/other)
    # newcmp is already defined in your environment
    
    # Map MTF values to colormap indices (0-5)
    def get_color(mtf):
        # MTF should be 1-5, use 0-4 for colormap, 5 for grey/other
        if mtf in [1, 2, 3, 4, 5]:
            idx = int(mtf) - 1
        else:
            idx = 5  # grey for missing/other
        rgba = newcmp(idx)
        # Convert RGBA (0-1) to hex
        return matplotlib.colors.to_hex(rgba)
    
    geojson_layers = []
    # Add each scenario's GeoDataFrame as a separate layer
    for n in range(1, n_scenario + 1):
        gdf = locals()[ccode + '_elec_SSP' + str(n) + '_2025'].copy()
        gdf = gdf.to_crs(epsg=4326)
        
        layer_name = f'SSP{n}, 2025'
        # Add a new column to gdf for tooltip header
        gdf['layer_name'] = layer_name
        
        gj = folium.GeoJson(
            gdf,
            name=layer_name,
            style_function=lambda feature: {
                'fillColor': get_color(feature['properties']['2025_MTF']),
                'color': get_color(feature['properties']['2025_MTF']),
                'weight': 0.05,
                'fillOpacity': 0.75,
            },
            tooltip=folium.GeoJsonTooltip(
                fields=['layer_name', '2025_POP', '2025_INC', '2025_elec', '2025_MTF', '2025_perCa', '2025_TotEl'],
                aliases=['Scenario:', 'Population:', 'Income level [$/capita]:', 'Electrification status [T/F]:', 'MTF class [0-5]:', 'Average electricity demand [kWh/capita]:', 'Total electricity demand [kWh]:'],
                labels=True
            ),
            show=(n == 1)  # Only the first layer is shown by default
        )
        gj.add_to(m)
        geojson_layers.append(gj)
    
    # Set collapsed=False to expand the layer list by default
    folium.LayerControl(collapsed=False).add_to(m)
    # Save the map to an HTML file
    m.save(ccode + '_SSPall_MTF.html')

In [ ]:
import webbrowser

webbrowser.open('map_SSP1.html', new=2)  # open in new tab